In [0]:
input_path = "/Volumes/workspace/default/dataframe-sr/"
checkpoint_location = "/Volumes/workspace/default/dataframe-sr/_checkpoint"

# Infer schema from the file first
df_static = spark.read.format("json").load(input_path + "sample.json")
inferred_schema = df_static.schema

# Use the schema for streaming with trigger(availableNow=True) for Serverless
df_stream = (spark.readStream
            .format("json")
            .schema(inferred_schema)
            .load(input_path)
)

query = (df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .toTable("workspace.default.dataframe_sr_output")
)

query.awaitTermination()

In [0]:
transformed_df = df_stream.withColumn(
    "bonus",
     df_stream.salary * 0.10
)

checkpoint_location_transformed = "/Volumes/workspace/default/dataframe-sr/_checkpoint_transformed"
display(transformed_df, checkpointLocation=checkpoint_location_transformed)

Checkpointing to /Volumes/workspace/default/dataframe-sr/_checkpoint_transformed


In [0]:
query = (
    transformed_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location_transformed)
    .trigger(availableNow=True)
    .toTable("workspace.default.dataframe_sr_output_transformed"))


In [0]:
%sql
SELECT *
FROM workspace.default.dataframe_sr_output

department,employee_id,name,salary
IT,101,John,60000
HR,102,Sarah,55000
IT,103,Mike,70000
Finance,104,Emma,65000
